In [ ]:
%load_ext autoreload
%autoreload 2
import itertools
import os

import torch
from compactreasoningmodels.datasets import NonogramDataset
from compactreasoningmodels.datasets.collate import collate_raw
from compactreasoningmodels.trace_comparison import Experiment, MSEBlock

if "original_dir" not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, ".."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "models")

In [ ]:
dataset = NonogramDataset("traces/large_dataset.jsonl")
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=1, shuffle=True, collate_fn=collate_raw
)
solver_list = list(next(iter(dataloader))[2][0]["traces"].keys())
print(solver_list)

In [ ]:
def make_pair_dataset(dataloader, traces):
    pair_dataset = []

    for X, y, meta in itertools.chain.from_iterable(
        zip(X_batch, y_batch, meta_batch) for X_batch, y_batch, meta_batch in dataloader
    ):
        pair = []
        for trace, idx in traces:
            pair.append(meta["traces"][trace][idx]["steps"])
        pair_dataset.append(pair)
    if len(pair_dataset) == 0:
        return None

    pair_dataset = torch.tensor(
        pair_dataset, dtype=torch.float32, device="cuda" if torch.cuda.is_available() else "cpu"
    ).permute(1, 0, 3, 4, 2)  # (traces, batch, height, width, steps)
    return pair_dataset

In [ ]:
# flatten = FlattenBlock(start_dim=2)
# cka = CKABlock()
# experiment = Experiment([flatten, cka], name="CKA_Comparison")
mse = MSEBlock()
experiment = Experiment([mse], name="MSE_Comparison")

In [ ]:
# One run per solver is generated, so there is no within-solver spread to
# measure; this cell is the seed for adding `n_runs` back to the generator.
for solver in solver_list:
    scores = []
    for idx1, idx2 in itertools.product(range(1), repeat=2):
        traces = [(solver, idx1), (solver, idx2)]
        pair_dataset = make_pair_dataset(dataloader, traces)
        if pair_dataset is None:
            continue
        result = experiment(pair_dataset)
        scores.append(result)
    adjusted = abs(torch.logit(torch.stack(scores).mean()))
    print(f"{solver:<30}{adjusted.item():.4f}")

In [ ]:
for solver1, solver2 in itertools.product(solver_list, repeat=2):
    traces = [(solver1, 0), (solver2, 0)]
    if solver1 == solver2 or solver1 < solver2:
        continue
    pair_dataset = make_pair_dataset(dataloader, traces)
    result = experiment(pair_dataset)
    adjusted = abs(torch.logit(result))
    print(f"{solver1:<30}{solver2:<30}{adjusted.item():.4f}")

In [ ]:
import itertools

import numpy as np
import pandas as pd
import torch

# Initialize lists to store results
results = []

for solver1, solver2 in itertools.product(solver_list, repeat=2):
    traces = [(solver1, 0), (solver2, 0)]

    if solver1 == solver2 or solver1 < solver2:
        continue

    pair_dataset = make_pair_dataset(dataloader, traces)
    result = experiment(pair_dataset)
    adjusted = abs(torch.logit(result))

    # Store results
    results.append({"solver1": solver1, "solver2": solver2, "adjusted_value": adjusted.item()})

# Create DataFrame from results
df = pd.DataFrame(results)

# Get all unique solvers
solvers = sorted(set(df["solver1"].tolist() + df["solver2"].tolist()))

# Create empty symmetric matrix
n = len(solvers)
symmetric_arr = np.full((n, n), np.nan)

# Fill in the values from the results
for _, row in df.iterrows():
    i = solvers.index(row["solver1"])
    j = solvers.index(row["solver2"])
    val = row["adjusted_value"]

    # Fill both positions for symmetry
    symmetric_arr[i, j] = val
    symmetric_arr[j, i] = val

# Create symmetric DataFrame
symmetric_df = pd.DataFrame(symmetric_arr, index=solvers, columns=solvers)

# Round for cleaner display
symmetric_df = symmetric_df.round(4)

# Display the symmetric matrix
print("Symmetric Solver Comparison Matrix:")
print(symmetric_df)

# Optional: Visualize as heatmap
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(12, 10))
    sns.heatmap(symmetric_df, annot=True, fmt=".4f", cmap="viridis", linewidths=0.5, square=True)
    plt.title("Symmetric Solver Comparison Matrix")
    plt.tight_layout()
    plt.savefig("symmetric_solver_heatmap.png", dpi=300, bbox_inches="tight")
    plt.show()
except ImportError:
    print("\nSeaborn/Matplotlib not available for visualization")

# Optional: Save to CSV
symmetric_df.to_csv("symmetric_solver_comparison.csv")
print("\nResults saved to 'symmetric_solver_comparison.csv'")

In [ ]:
df

In [ ]:
all_pairs = {}
for pair in itertools.combinations(solver_list, r=2):
    traces = [(pair[0], 0), (pair[1], 0)]
    pair_dataset = make_pair_dataset(dataloader, traces)
    result = experiment(pair_dataset)
    adjusted = abs(torch.logit(result))
    all_pairs[pair] = adjusted.item()
all_pairs

In [ ]:
# There is no longer a within-solver reference pair to compare against, so the
# triangle-inequality violation check that lived here has no baseline. It returns
# if `generate_traces` ever emits more than one run per solver again.
n_runs = len(next(iter(dataloader))[2][0]["traces"][solver_list[0]])
print(n_runs)